[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S12/S12_05_transformers_atencion.ipynb)

# S12 · 05 · Transformers y LLM: tokenización y atención paso a paso

**Tiempo estimado:** 1 h 10 min · **Sprint 12** (UD 4.9) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. Qué es un **token** y cómo trocea un texto un modelo de lenguaje (idea de BPE).
2. Cómo funciona la **autoatención** (*self-attention*), calculada a mano con `numpy` y números pequeños.
3. Qué hace la **máscara causal** y por qué un LLM genera palabra a palabra; y qué controla la **temperatura**.
4. (Opcional, con internet) Usar modelos preentrenados de Hugging Face: sentimiento y clasificación *zero-shot*.

**Contexto.** Antes de que TurisData use un LLM como asistente, su equipo debe entender qué hay dentro de la "caja negra":
no es magia, es **matrices y probabilidades**.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import re, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
np.set_printoptions(precision=2, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False, "axes.spines.right": False})

## 1. Tokenización: cómo "lee" un LLM
Un LLM no lee letras ni palabras completas, sino **tokens**: trozos frecuentes de texto.
- Trocear por **palabras** genera vocabularios enormes y no sabe qué hacer con palabras nuevas.
- Trocear por **letras** da secuencias larguísimas.
- **BPE** (*byte pair encoding*) es un punto medio: parte de letras y va **fusionando los pares más frecuentes**
  hasta formar trozos útiles ("desayun" + "o").

Vamos a implementar un BPE de juguete con un mini corpus.

In [ ]:
corpus = ["desayuno", "desayunos", "desayunar", "limpio", "limpieza", "limpiar", "limpiamos", "desayunamos"]

def contar_pares(palabras):
    """palabras: lista de listas de símbolos. Devuelve un Counter con la frecuencia de cada par de símbolos consecutivos."""
    pares = Counter()
    for simbolos in palabras:
        for par in zip(simbolos, simbolos[1:]):
            pares[par] += 1
    return pares

**Ejercicio 1.** Comprueba que `contar_pares` funciona: calcula `pares` sobre las palabras del corpus troceadas en letras
(`[list(p) for p in corpus]`) y guarda en `par_mas_comun` el par más frecuente (una tupla de 2 letras).

In [ ]:
# TODO: convierte el corpus a listas de letras y usa most_common(1)
...
print("Par más común:", par_mas_comun, "→", pares[par_mas_comun], "veces")
assert isinstance(par_mas_comun, tuple) and len(par_mas_comun) == 2, "par_mas_comun debe ser una tupla de dos símbolos, p. ej. ('a', 'r')."
assert pares[par_mas_comun] == max(pares.values()), "Debe ser el par con más apariciones."

In [ ]:
def entrenar_bpe(palabras, n_fusiones):
    """Aplica n_fusiones: en cada paso fusiona el par más frecuente en un solo símbolo."""
    palabras = [list(p) for p in palabras]
    fusiones = []
    for _ in range(n_fusiones):
        pares = contar_pares(palabras)
        if not pares:
            break
        a, b = pares.most_common(1)[0][0]
        fusiones.append((a, b))
        nuevas = []
        for simbolos in palabras:
            i, salida = 0, []
            while i < len(simbolos):
                if i < len(simbolos) - 1 and simbolos[i] == a and simbolos[i + 1] == b:
                    salida.append(a + b); i += 2
                else:
                    salida.append(simbolos[i]); i += 1
            nuevas.append(salida)
        palabras = nuevas
    return fusiones, palabras

fusiones, tokens = entrenar_bpe(corpus, 12)
print("Fusiones aprendidas:", fusiones)
for p, t in zip(corpus, tokens):
    print(f"{p:12s} → {t}")

Observa cómo aparecen trozos como `desayun` o `limpi`: el modelo ha "descubierto" raíces sin saber español.
Una palabra nueva como "desayunando" se trocearía reutilizando esos trozos.

**Ejercicio 2.** Cuenta cuántos tokens hay en total tras las 12 fusiones (`n_tokens_bpe`) y cuántas letras tenía el corpus (`n_letras`).
¿Cuál es la **razón de compresión** (`n_letras / n_tokens_bpe`, redondeada a 2 decimales) en `compresion`?

In [ ]:
# TODO: suma de len() de cada lista de tokens; suma de len() de cada palabra original
...
print(f"{n_letras} letras → {n_tokens_bpe} tokens (compresión {compresion}x)")
assert n_letras == sum(len(p) for p in corpus), "n_letras es la suma de letras de las palabras del corpus."
assert n_tokens_bpe < n_letras, "Con las fusiones debe haber menos tokens que letras."
assert compresion > 1, "compresion = n_letras / n_tokens_bpe debe ser mayor que 1."

**Por qué importa en la práctica:** los LLM cobran y se limitan **por tokens**, y el español suele gastar más tokens que el inglés.

## 2. Autoatención paso a paso (con números pequeños)
**Idea.** Al leer "El hotel es limpio pero ruidoso", para entender *ruidoso* conviene mirar a *hotel*.
La **atención** deja que cada palabra decida **cuánto mirar a las demás** y mezcle su información.
Se hace con tres versiones de cada palabra:
- **Q (query, "pregunta")**: lo que esta palabra busca.
- **K (key, "etiqueta")**: lo que cada palabra ofrece.
- **V (value, "contenido")**: la información que se pasa.

Pasos: 1) Q·Kᵀ = *puntuaciones* de compatibilidad; 2) dividir por √d; 3) **softmax** por filas → pesos que suman 1;
4) salida = pesos · V.

Usamos **3 palabras** y vectores de tamaño 4, con números enteros para poder seguir las cuentas a mano.

In [ ]:
palabras = ["hotel", "muy", "limpio"]
# Embeddings inventados (una fila por palabra, 4 dimensiones)
X = np.array([[1., 0., 1., 0.],
              [0., 2., 0., 2.],
              [1., 1., 0., 1.]])

# Matrices de proyección (4 → 2). En un modelo real se APRENDEN durante el entrenamiento.
Wq = np.array([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
Wk = np.array([[1., 0.], [0., 1.], [0., 1.], [1., 0.]])
Wv = np.array([[1., 2.], [0., 1.], [1., 0.], [0., 1.]])

Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("Q =\n", Q, "\nK =\n", K, "\nV =\n", V)

**Paso 1: puntuaciones** = Q · Kᵀ. La fila *i*, columna *j* dice cuánto le interesa la palabra *i* la palabra *j*.

In [ ]:
puntuaciones = Q @ K.T
print(puntuaciones)

**Paso 2: escalar** por √d (d = 2 aquí) para que los números no crezcan demasiado, y **paso 3: softmax**.
El softmax convierte cada fila en probabilidades (positivas, suman 1): `exp(x) / suma(exp(x))`.

**Ejercicio 3.** Escribe `softmax(m)` que aplique el softmax **por filas** a una matriz (resta el máximo de cada fila antes
de exponenciar, por estabilidad numérica).

In [ ]:
def softmax(m):
    # TODO: resta el máximo de cada fila (keepdims=True), aplica exp y divide cada fila entre su suma
    ...

_p = softmax(np.array([[1., 2., 3.], [0., 0., 0.]]))
assert np.allclose(_p.sum(axis=1), 1), "Cada fila de softmax debe sumar 1 (usa axis=1)."
assert np.allclose(_p[1], 1 / 3), "Con entradas iguales, todas las probabilidades valen lo mismo."
assert _p[0, 2] > _p[0, 1] > _p[0, 0], "Un valor mayor debe recibir más probabilidad."

In [ ]:
d = K.shape[1]
pesos = softmax(puntuaciones / np.sqrt(d))
print("Pesos de atención (cada fila suma 1):\n", pesos)

**Paso 4: salida** = pesos · V. Cada palabra se convierte en una **mezcla** de los contenidos de las demás, según sus pesos.
Ese vector ya "sabe" del contexto.

**Ejercicio 4.** Completa `atencion(X, Wq, Wk, Wv)` que devuelve `(salida, pesos)` siguiendo los 4 pasos.

In [ ]:
def atencion(X, Wq, Wk, Wv):
    # TODO: sigue los 4 pasos: proyecciones, puntuaciones escaladas, softmax y mezcla de V
    ...

salida, pesos2 = atencion(X, Wq, Wk, Wv)
print("Salida:\n", salida)
assert salida.shape == (3, 2), "La salida debe tener una fila por palabra y 2 columnas (dimensión de V)."
assert np.allclose(pesos2, pesos), "Los pesos deben coincidir con los calculados paso a paso arriba."
assert np.allclose(salida, pesos @ V), "salida = pesos @ V."

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(pesos, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(palabras); ax.set_yticks(range(3)); ax.set_yticklabels(palabras)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{pesos[i, j]:.2f}", ha="center", va="center", color="black" if pesos[i, j] < .6 else "white")
ax.set_xlabel("Mira a…"); ax.set_ylabel("Palabra que pregunta"); ax.set_title("Pesos de atención")
plt.tight_layout(); plt.show()

## 3. Máscara causal: mirar solo hacia atrás
Un LLM genera texto **de izquierda a derecha**: al predecir la palabra 2, no puede ver la 3 (aún no existe).
Se consigue **poniendo −∞ en las puntuaciones de las palabras futuras** antes del softmax: su peso pasa a 0.

**Ejercicio 5.** Completa `atencion_causal(X, Wq, Wk, Wv)`: igual que `atencion`, pero antes del softmax
pon `-1e9` en las posiciones **por encima de la diagonal** (`np.triu(..., k=1)`).

In [ ]:
def atencion_causal(X, Wq, Wk, Wv):
    # TODO: igual que atencion, pero con una máscara booleana del triángulo superior (np.triu, k=1) puesta a -1e9 antes del softmax
    ...

_, pesos_c = atencion_causal(X, Wq, Wk, Wv)
print(pesos_c)
assert np.allclose(np.triu(pesos_c, k=1), 0), "Los pesos por encima de la diagonal (futuro) deben ser 0."
assert np.allclose(pesos_c.sum(axis=1), 1), "Cada fila sigue sumando 1."
assert np.isclose(pesos_c[0, 0], 1.0), "La primera palabra solo puede mirarse a sí misma."

## 4. Del cálculo a la generación: logits, softmax y temperatura
Al final, un LLM produce para cada posible siguiente token una puntuación (**logit**). El softmax las convierte en
probabilidades y se **elige** una. La **temperatura** T reparte esas probabilidades: T baja → siempre lo más probable (seguro, repetitivo);
T alta → más variedad (creativo, más riesgo de errores/alucinaciones).

In [ ]:
siguiente = ["limpio", "ruidoso", "caro", "cerca", "agradable"]
logits = np.array([[3.0, 1.0, 0.5, 2.0, 2.5]])
fig, ejes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, T in zip(ejes, [0.3, 1.0, 2.5]):
    p = softmax(logits / T)[0]
    ax.bar(siguiente, p, color="#3b7ea1"); ax.set_title(f"Temperatura {T}"); ax.tick_params(axis="x", rotation=45)
ejes[0].set_ylabel("Probabilidad")
plt.tight_layout(); plt.show()

**Ejercicio 6.** Calcula `prob_max_T03` (la probabilidad del token más probable con `T = 0.3`) y `prob_max_T25` (con `T = 2.5`).
¿Cuál de las dos es mayor? Escribe la respuesta en `mas_seguro`: `"T baja"` o `"T alta"`.

In [ ]:
# TODO: softmax(logits / T) y .max(); la temperatura baja concentra la probabilidad
...
print(round(prob_max_T03, 3), round(prob_max_T25, 3), mas_seguro)
assert 0 < prob_max_T25 < prob_max_T03 <= 1, "Con T más baja la distribución es más puntiaguda: prob_max_T03 debe ser mayor."
assert mas_seguro == "T baja", "mas_seguro: la temperatura baja da respuestas más predecibles."

## 5. Mini-reto integrador: ¿de qué depende que un LLM "mire" bien?
Con las mismas matrices Wq, Wk y Wv, modifica el embedding de "limpio" para que la palabra "hotel" le **preste más atención**.
Crea `X2` (una copia de `X` con la fila de "limpio", índice 2, cambiada) para que `pesos_nuevos[0, 2]` (cuánto mira "hotel" a "limpio") sea **mayor** que `pesos[0, 2]`
(la atención original). Guarda en `subio` (True/False) el resultado de la comparación.

In [ ]:
# TODO: multiplica por 2 la fila de "limpio" en X2: sus claves K aumentan y hotel le presta más atención
...
print("Antes:", round(float(pesos[0, 2]), 3), "| Ahora:", round(float(pesos_nuevos[0, 2]), 3))
assert X2.shape == X.shape, "X2 debe tener la misma forma que X."
assert not np.allclose(X2, X), "Modifica X2 para que no sea igual a X."
assert subio, "Ajusta X2 hasta que hotel preste más atención a limpio que antes (pesos_nuevos[0, 2] > pesos[0, 2])."

## 6. OPCIONAL · Hugging Face en Colab: sentimiento y *zero-shot* (requiere internet)
> **⚠️ NO VERIFICADO EN EL TALLER:** necesita instalar `transformers`, descargar modelos de Hugging Face (cientos de MB)
> y conexión a internet. Ejecútalo en **Google Colab** (mejor con GPU: *Entorno de ejecución → Cambiar tipo → GPU T4*).
> Los nombres de modelo pueden cambiar con el tiempo: si uno falla, busca uno equivalente en huggingface.co/models.

- **`sentiment-analysis`**: clasifica el sentimiento con un modelo ya entrenado (sin entrenar nada tú).
- **`zero-shot-classification`**: clasifica un texto en **etiquetas que tú inventas** sin ejemplos de entrenamiento.
- **Fine-tuning ligero** (idea): en vez de reentrenar todo el modelo, se ajusta una parte pequeña (p. ej. capas finales o adaptadores LoRA)
  con tus propios ejemplos. Se ve en teoría; aquí no se ejecuta.

In [ ]:
USAR_HUGGING_FACE = False   # ← True solo en Colab con internet

if USAR_HUGGING_FACE:
    # NO VERIFICADO: requiere internet y descarga de modelos
    %pip install -q transformers
    from transformers import pipeline
    opiniones = ["El desayuno era delicioso y el personal encantador", "Ruido toda la noche y nadie atendía en recepción"]
    clasificador = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    print(clasificador(opiniones))
    zero = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
    print(zero("Nadie atendía en recepción", candidate_labels=["limpieza", "personal", "ruido", "precio"],
               hypothesis_template="Esta reseña trata de {}."))
else:
    print("Parte opcional omitida (USAR_HUGGING_FACE = False).")

## Preguntas de reflexión
1. En el ejercicio de la atención, ¿qué representan las filas y qué las columnas de la matriz de pesos? ¿Por qué cada fila suma 1?
2. ¿Por qué un LLM con temperatura alta puede "inventar" datos (alucinar) con más facilidad?
3. Un modelo cobra por tokens y el español consume más que el inglés: ¿cómo afectaría al coste de un asistente para el cliente?
4. Si usas un modelo preentrenado con `zero-shot`, ¿qué comprobarías antes de fiarte de él con las reseñas reales de TurisData?

## Resumen: qué has aprendido
- Un LLM trabaja con **tokens** (trozos de texto), obtenidos con métodos como BPE.
- La **autoatención** mezcla la información de todas las palabras según pesos calculados con Q, K y V; un Transformer repite esto en muchas capas y "cabezas".
- La **máscara causal** evita mirar al futuro; la **temperatura** regula la variedad de lo que se genera.
- Los modelos preentrenados (Hugging Face) permiten sentimiento y clasificación *zero-shot* sin entrenar, pero hay que **evaluarlos** con tus datos.